In [141]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2023-11-21 19:29:12.565113: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-11-21 19:29:12.801956: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-11-21 19:29:12.805924: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-21 19:29:14.624636: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


matrix_approx_zeshel.py


# Open Data loader & context

In [142]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [143]:
from sklearn.cluster import KMeans

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        #np.random.shuffle(self.requests)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

# Games Data loader & context

In [144]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

# Models

In [145]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [146]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [147]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [148]:
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# Evals

In [149]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

def K_by_min(X):
    center = X.mean(axis=0)
    K = [euclidean_distances(X, center.reshape(1, -1)).argmax()]

    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def eval_clustering(ctx, all_from_labels=False):
    X = ctx.get_relevs("train").T
    from sklearn.cluster import KMeans
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    
    K_KMeans = k_func(
        KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )


    from sklearn.cluster import BisectingKMeans
    K_BisectingKMeans = k_func(
        BisectingKMeans(n_clusters=100, random_state=0).fit(X)
    )


    from sklearn.cluster import MiniBatchKMeans
    K_MiniBatchKMeans = from_labels(
        X,
        MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
    )

    from sklearn.cluster import AgglomerativeClustering
    K_AgglomerativeClustering = from_labels(
        X,
        AgglomerativeClustering(n_clusters=100).fit(X).labels_
    )

    from sklearn.cluster import SpectralCoclustering
    K_SpectralCoclustering = from_labels(
        X,
        SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralBiclustering
    K_SpectralBiclustering = from_labels(
        X,
        SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralClustering
    K_SpectralClusteringNN = from_labels(
        X,
        SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
    )

    K_ByMin = K_by_min(X)

    ev([
        Popular(ctx),
        AnnCUR(ctx),
        AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
        AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
        AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
        AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans"),
        AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
        AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
        AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
        AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
        AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
        AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
    ])

In [150]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

# support items choice

In [151]:
# setup:
# 1000 items, 1000 + 1000 queries
recall_k = 5
n_runs = 10
n_items_per_run = 1000
test_item_cnt = 100
item_cnt_range = list(range(5, 101, 5))
assert test_item_cnt in item_cnt_range

In [152]:
from collections import defaultdict
from itertools import combinations

from matplotlib import pyplot as plt
import numpy as np
import os
from sklearn.metrics import top_k_accuracy_score
from sklearn.cluster import KMeans
from scipy.stats import mannwhitneyu, ttest_rel

In [153]:
#with open("collections_10000_items/test.npy", "rb") as fin:
#    all_test = np.load(fin)
#with open("collections_10000_items/train.npy", "rb") as fin:
#    all_train = np.load(fin)

#assert all_train.shape[0] == n_runs * n_items_per_run
#assert all_test.shape[0] == n_runs * n_items_per_run


In [154]:
def eval_items(items, train, test):
    item_embeds = train.dot(np.linalg.pinv(train[items]))
    approx_test_rel = item_embeds.dot(test[items])
    approx_train_rel = item_embeds.dot(train[items])
    best_items = np.argsort(test, axis=0)[-1]
    recall = top_k_accuracy_score(best_items, approx_test_rel.T, k=recall_k, labels=np.arange(train.shape[0]))
    test_mse = np.mean((test - approx_test_rel) ** 2)
    train_mse = np.mean((train - approx_train_rel) ** 2)
    return recall, test_mse, train_mse

def get_rnd_items(n_items, train):
    return np.random.choice(train.shape[0], n_items, replace=False)

def get_kmeans_items(n_items, train):
    kmeans = KMeans(n_clusters=n_items, n_init="auto").fit(train)
    items = []
    for center in kmeans.cluster_centers_:
        distances = ((train - center) ** 2).sum(axis=1)
        assert distances.shape == (train.shape[0],)
        for item_id in np.argsort(distances):
            if item_id not in items:
                items.append(item_id)
                break
    return np.array(items, dtype=np.int64)

greedy_items = []
def get_greedy(n_items, train, lazy=True):
    global greedy_items
    if not lazy:
        greedy_items = []
    if len(greedy_items) >= n_items:
        return np.array(greedy_items[:n_items], dtype=np.int64)
    
    gram = train.dot(train.T)
    gram /= gram.mean()
    items = []
    remaining_items = np.ones(train.shape[0], dtype="bool")
    for t in range(n_items):
        scores = (gram ** 2).sum(axis=1)
        assert np.allclose(scores[~remaining_items], np.zeros_like(scores[~remaining_items])), (
            t, scores[~remaining_items]
        )
        scores[remaining_items] /= gram[remaining_items, remaining_items]
        if max(scores) < 1e-9:
            break
        new_item = scores.argmax()
        assert remaining_items[new_item]
        coefs = gram[new_item] / np.sqrt(gram[new_item, new_item])
        diff = coefs.reshape(-1, 1).dot(coefs.reshape(1, -1))
        gram -= diff
        assert np.allclose(gram[new_item], np.zeros_like(gram[new_item]), atol=1e-6), (
            t, gram[new_item].std()
        )
        items.append(new_item)
        remaining_items[new_item] = False
    assert len(items) == n_items
    greedy_items.clear()
    greedy_items.extend(items)
    return np.array(items, dtype=np.int64)


### fast-a

In [157]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    print(candidate_items.shape, target_items.shape)
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        print(candidate_coitems.shape, candidate_items.shape)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

In [158]:
import cProfile as profile
import pstats

In [159]:
dataset = DATASETS[0]

print ("\n\n\nDATASET = ", dataset)

ctx = EvalContextRelevs(
    load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
    det_attempts=100
)

print("LOADED")

# ctx.set_kmeans_games_as_key()


t = ctx.get_relevs("train").T
t = (t - t.mean()) / t.std()
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED
(10031, 2260) (10031, 2260)


/tmp/ipykernel_919264/3953128763.py:16: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

(10031, 2260) (10031, 2260)
(10031, 2260) (10031, 2260)
(10031, 2260) (10031, 2260)
(10031, 2260) (10031, 2260)
(10031, 2260) (10031, 2260)
(10031, 2260) (10031, 2260)
(10031, 2260) (10031, 2260)
(10031, 2260) (10031, 2260)


KeyboardInterrupt: 

In [ ]:
N = 1000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 100,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005
})

p = profile.Profile()
p.enable()
m.fit()
p.disable()

pstats.Stats(p).sort_stats('cumulative').print_stats(30)

In [24]:
tf.config.threading.get_intra_op_parallelism_threads()

0

In [29]:
pstats.Stats(p).sort_stats('cumtime').print_stats(30)

         7154328 function calls (6943687 primitive calls) in 73.284 seconds

   Ordered by: cumulative time
   List reduced from 1427 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000   73.284   36.642 /home/shevkunov/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3333(run_code)
        2    0.000    0.000   73.284   36.642 {built-in method builtins.exec}
        1    0.001    0.001   73.284   73.284 /var/tmp/ipykernel_820429/2471865983.py:23(<cell line: 23>)
        1    0.037    0.037   73.283   73.283 /var/tmp/ipykernel_820429/2783432684.py:133(fit)
       30    1.907    0.064   56.029    1.868 /var/tmp/ipykernel_820429/1883765668.py:25(get_score)
27051/19500    0.329    0.000   53.702    0.003 {built-in method numpy.core._multiarray_umath.implement_array_function}
      363    0.003    0.000   49.121    0.135 /home/shevkunov/anaconda3/lib/python3.9/site-packages/numpy/core/fromn

In [30]:
print(np.show_config())

openblas64__info:
    libraries = ['openblas64_', 'openblas64_']
    library_dirs = ['/usr/local/lib']
    language = c
    define_macros = [('HAVE_CBLAS', None), ('BLAS_SYMBOL_SUFFIX', '64_'), ('HAVE_BLAS_ILP64', None)]
    runtime_library_dirs = ['/usr/local/lib']
blas_ilp64_opt_info:
    libraries = ['openblas64_', 'openblas64_']
    library_dirs = ['/usr/local/lib']
    language = c
    define_macros = [('HAVE_CBLAS', None), ('BLAS_SYMBOL_SUFFIX', '64_'), ('HAVE_BLAS_ILP64', None)]
    runtime_library_dirs = ['/usr/local/lib']
openblas64__lapack_info:
    libraries = ['openblas64_', 'openblas64_']
    library_dirs = ['/usr/local/lib']
    language = c
    define_macros = [('HAVE_CBLAS', None), ('BLAS_SYMBOL_SUFFIX', '64_'), ('HAVE_BLAS_ILP64', None), ('HAVE_LAPACKE', None)]
    runtime_library_dirs = ['/usr/local/lib']
lapack_ilp64_opt_info:
    libraries = ['openblas64_', 'openblas64_']
    library_dirs = ['/usr/local/lib']
    language = c
    define_macros = [('HAVE_CBLAS', None

In [37]:
import mkl
mkl.set_num_threads(32), mkl.get_max_threads()

(32, 32)

In [26]:
%%time

ma = AnnCUR(ctx)
ma.fit()
ma.get_score("test")

N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.001,
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False
})

m.fit()

np.mean(results), mse, len(results) =  0.5617472852912143 0.146082 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.1632 tf.Tensor(94.47289408248253, shape=(), dtype=float64) 100
slice = key, score = 0.1632
np.mean(results), mse, len(results) =  0.16001769911504424 tf.Tensor(93.25397247400434, shape=(), dtype=float64) 2260
slice = train, score = 0.16001769911504424
np.mean(results), mse, len(results) =  0.15115498519249754 tf.Tensor(93.51968379546467, shape=(), dtype=float64) 1013
slice = test, score = 0.15115498519249754

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.4652000000000001 tf.Tensor(159.25217629346264, shape=(), dtype=float64) 100
slice = key, score = 0.4652000000000001
np.mean(results),

np.mean(results), mse, len(results) =  0.5080530973451327 tf.Tensor(1661.9709438396005, shape=(), dtype=float64) 2260
slice = train, score = 0.5080530973451327
np.mean(results), mse, len(results) =  0.48611056268509373 tf.Tensor(1645.6398658250548, shape=(), dtype=float64) 1013
slice = test, score = 0.48611056268509373

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.5334 tf.Tensor(1690.39459421561, shape=(), dtype=float64) 100
slice = key, score = 0.5334
np.mean(results), mse, len(results) =  0.5350309734513274 tf.Tensor(1737.1701521896075, shape=(), dtype=float64) 2260
slice = train, score = 0.5350309734513274
np.mean(results), mse, len(results) =  0.5138499506416584 tf.Tensor(1715.3891685816093, shape=(), dtype=float64) 1013
slice = test, score = 0.5138499506416584

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5246000000000001 tf.Tensor(1449.679189296694, shape=(), dtype=float64) 100
slice = key, score = 0.5246000000000001
np.mean(results), mse, 

np.mean(results), mse, len(results) =  0.5547876106194691 tf.Tensor(1251.278832180556, shape=(), dtype=float64) 2260
slice = train, score = 0.5547876106194691
np.mean(results), mse, len(results) =  0.5270187561697927 tf.Tensor(1238.1710645653407, shape=(), dtype=float64) 1013
slice = test, score = 0.5270187561697927

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.5243 tf.Tensor(1307.935171723496, shape=(), dtype=float64) 100
slice = key, score = 0.5243
np.mean(results), mse, len(results) =  0.527 tf.Tensor(1307.0342419824635, shape=(), dtype=float64) 2260
slice = train, score = 0.527
np.mean(results), mse, len(results) =  0.5028035538005924 tf.Tensor(1288.5828712272212, shape=(), dtype=float64) 1013
slice = test, score = 0.5028035538005924

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.5454 tf.Tensor(1355.6677717693535, shape=(), dtype=float64) 100
slice = key, score = 0.5454
np.mean(results), mse, len(results) =  0.5544513274336283 tf.Tensor(1365.0

np.mean(results), mse, len(results) =  0.5637035398230089 tf.Tensor(1386.0017890186168, shape=(), dtype=float64) 2260
slice = train, score = 0.5637035398230089
np.mean(results), mse, len(results) =  0.5330898321816386 tf.Tensor(1454.1763891724613, shape=(), dtype=float64) 1013
slice = test, score = 0.5330898321816386

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.5617000000000002 tf.Tensor(1123.2658476622498, shape=(), dtype=float64) 100
slice = key, score = 0.5617000000000002
np.mean(results), mse, len(results) =  0.567858407079646 tf.Tensor(1217.5725623233002, shape=(), dtype=float64) 2260
slice = train, score = 0.567858407079646
np.mean(results), mse, len(results) =  0.5368015794669299 tf.Tensor(1254.9642884587652, shape=(), dtype=float64) 1013
slice = test, score = 0.5368015794669299

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.5665 tf.Tensor(1172.1809782219975, shape=(), dtype=float64) 100
slice = key, score = 0.5665
np.mean(results), mse, l

np.mean(results), mse, len(results) =  0.534471865745311 tf.Tensor(3517.960734898203, shape=(), dtype=float64) 1013
slice = test, score = 0.534471865745311

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.5691999999999999 tf.Tensor(2070.9222180290285, shape=(), dtype=float64) 100
slice = key, score = 0.5691999999999999
np.mean(results), mse, len(results) =  0.5738893805309735 tf.Tensor(2066.578018514797, shape=(), dtype=float64) 2260
slice = train, score = 0.5738893805309735
np.mean(results), mse, len(results) =  0.5419842053307009 tf.Tensor(2139.7373996922356, shape=(), dtype=float64) 1013
slice = test, score = 0.5419842053307009

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.5774 tf.Tensor(2416.654419849326, shape=(), dtype=float64) 100
slice = key, score = 0.5774
np.mean(results), mse, len(results) =  0.5776637168141593 tf.Tensor(2402.688206987981, shape=(), dtype=float64) 2260
slice = train, score = 0.5776637168141593
np.mean(results), mse, len(r

np.mean(results), mse, len(results) =  0.5367226061204344 tf.Tensor(8276.129725881416, shape=(), dtype=float64) 1013
slice = test, score = 0.5367226061204344

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.5748 tf.Tensor(9965.257904121605, shape=(), dtype=float64) 100
slice = key, score = 0.5748
np.mean(results), mse, len(results) =  0.5825088495575221 tf.Tensor(9782.290726047831, shape=(), dtype=float64) 2260
slice = train, score = 0.5825088495575221
np.mean(results), mse, len(results) =  0.5479664363277393 tf.Tensor(10163.583928921733, shape=(), dtype=float64) 1013
slice = test, score = 0.5479664363277393

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.5721000000000002 tf.Tensor(8799.612042904828, shape=(), dtype=float64) 100
slice = key, score = 0.5721000000000002
np.mean(results), mse, len(results) =  0.5755353982300885 tf.Tensor(8677.15036914275, shape=(), dtype=float64) 2260
slice = train, score = 0.5755353982300885
np.mean(results), mse, len(r

In [29]:
%%time

ma = AnnCUR(ctx)
ma.fit()
ma.get_score("test")

N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005,
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False
})

m.fit()

np.mean(results), mse, len(results) =  0.5617472852912143 0.146082 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.08410000000000001 tf.Tensor(88.76756870053032, shape=(), dtype=float64) 100
slice = key, score = 0.08410000000000001
np.mean(results), mse, len(results) =  0.07862389380530974 tf.Tensor(89.09198918720573, shape=(), dtype=float64) 2260
slice = train, score = 0.07862389380530974
np.mean(results), mse, len(results) =  0.07976307996051334 tf.Tensor(89.12761961048294, shape=(), dtype=float64) 1013
slice = test, score = 0.07976307996051334

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.45710000000000006 tf.Tensor(131.6627168488877, shape=(), dtype=float64) 100
slice = key, score = 0.45710000

np.mean(results), mse, len(results) =  0.5254787759131293 tf.Tensor(1109.6475052978485, shape=(), dtype=float64) 1013
slice = test, score = 0.5254787759131293

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.5442 tf.Tensor(1096.3770304250793, shape=(), dtype=float64) 100
slice = key, score = 0.5442
np.mean(results), mse, len(results) =  0.544924778761062 tf.Tensor(1107.524283681283, shape=(), dtype=float64) 2260
slice = train, score = 0.544924778761062
np.mean(results), mse, len(results) =  0.5185291214215202 tf.Tensor(1103.3438230047584, shape=(), dtype=float64) 1013
slice = test, score = 0.5185291214215202

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5480999999999999 tf.Tensor(1268.934910889127, shape=(), dtype=float64) 100
slice = key, score = 0.5480999999999999
np.mean(results), mse, len(results) =  0.5527433628318583 tf.Tensor(1287.5093099287533, shape=(), dtype=float64) 2260
slice = train, score = 0.5527433628318583
np.mean(results), mse, len

np.mean(results), mse, len(results) =  0.5185389930898322 tf.Tensor(2049.425673933988, shape=(), dtype=float64) 1013
slice = test, score = 0.5185389930898322

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.5663 tf.Tensor(2052.3940242016674, shape=(), dtype=float64) 100
slice = key, score = 0.5663
np.mean(results), mse, len(results) =  0.571491150442478 tf.Tensor(2095.8428347003965, shape=(), dtype=float64) 2260
slice = train, score = 0.571491150442478
np.mean(results), mse, len(results) =  0.5401480750246792 tf.Tensor(2082.6033890078206, shape=(), dtype=float64) 1013
slice = test, score = 0.5401480750246792

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.5675 tf.Tensor(2248.0220305698326, shape=(), dtype=float64) 100
slice = key, score = 0.5675
np.mean(results), mse, len(results) =  0.5722300884955752 tf.Tensor(2324.077696645553, shape=(), dtype=float64) 2260
slice = train, score = 0.5722300884955752
np.mean(results), mse, len(results) =  0.543089832

np.mean(results), mse, len(results) =  0.5421322803553801 tf.Tensor(2803.2326580950034, shape=(), dtype=float64) 1013
slice = test, score = 0.5421322803553801

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.5762 tf.Tensor(2921.9557496703474, shape=(), dtype=float64) 100
slice = key, score = 0.5762
np.mean(results), mse, len(results) =  0.5807610619469027 tf.Tensor(2973.6719839803764, shape=(), dtype=float64) 2260
slice = train, score = 0.5807610619469027
np.mean(results), mse, len(results) =  0.547482724580454 tf.Tensor(2973.924251393879, shape=(), dtype=float64) 1013
slice = test, score = 0.547482724580454

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.5772 tf.Tensor(2503.545559755417, shape=(), dtype=float64) 100
slice = key, score = 0.5772
np.mean(results), mse, len(results) =  0.5816725663716814 tf.Tensor(2544.0341051801715, shape=(), dtype=float64) 2260
slice = train, score = 0.5816725663716814
np.mean(results), mse, len(results) =  0.548884501

np.mean(results), mse, len(results) =  0.550019743336624 tf.Tensor(3196.5939696865057, shape=(), dtype=float64) 1013
slice = test, score = 0.550019743336624

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.5875 tf.Tensor(3177.3540944809265, shape=(), dtype=float64) 100
slice = key, score = 0.5875
np.mean(results), mse, len(results) =  0.5890840707964602 tf.Tensor(3178.363307946662, shape=(), dtype=float64) 2260
slice = train, score = 0.5890840707964602
np.mean(results), mse, len(results) =  0.5540572556762092 tf.Tensor(3201.726560557143, shape=(), dtype=float64) 1013
slice = test, score = 0.5540572556762092

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.5731 tf.Tensor(2901.360525669232, shape=(), dtype=float64) 100
slice = key, score = 0.5731
np.mean(results), mse, len(results) =  0.5823849557522123 tf.Tensor(2922.005585814119, shape=(), dtype=float64) 2260
slice = train, score = 0.5823849557522123
np.mean(results), mse, len(results) =  0.54813425468

np.mean(results), mse, len(results) =  0.550700888450148 tf.Tensor(3746.5272270702458, shape=(), dtype=float64) 1013
slice = test, score = 0.550700888450148

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.5767 tf.Tensor(3581.881621949247, shape=(), dtype=float64) 100
slice = key, score = 0.5767
np.mean(results), mse, len(results) =  0.5867300884955753 tf.Tensor(3528.31244704998, shape=(), dtype=float64) 2260
slice = train, score = 0.5867300884955753
np.mean(results), mse, len(results) =  0.5508588351431392 tf.Tensor(3566.558938614467, shape=(), dtype=float64) 1013
slice = test, score = 0.5508588351431392

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.5833999999999999 tf.Tensor(3242.1787947383095, shape=(), dtype=float64) 100
slice = key, score = 0.5833999999999999
np.mean(results), mse, len(results) =  0.5888716814159293 tf.Tensor(3161.9446327938836, shape=(), dtype=float64) 2260
slice = train, score = 0.5888716814159293
np.mean(results), mse, len(r

In [30]:
%%time

ma = AnnCUR(ctx)
ma.fit()
ma.get_score("test")

N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005,
    "normalize_embs": False
})

m.fit()

np.mean(results), mse, len(results) =  0.5617472852912143 0.146082 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0483 tf.Tensor(9.9987, shape=(), dtype=float32) 100
slice = key, score = 0.0483
np.mean(results), mse, len(results) =  0.04760176991150443 tf.Tensor(10.045445, shape=(), dtype=float32) 2260
slice = train, score = 0.04760176991150443
np.mean(results), mse, len(results) =  0.046347482724580454 tf.Tensor(10.126621, shape=(), dtype=float32) 1013
slice = test, score = 0.046347482724580454

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.09239999999999998 tf.Tensor(613.3464, shape=(), dtype=float32) 100
slice = key, score = 0.09239999999999998
np.mean(results), mse, len(results) =  0.088606194

np.mean(results), mse, len(results) =  0.5148849557522124 tf.Tensor(1841.4346, shape=(), dtype=float32) 2260
slice = train, score = 0.5148849557522124
np.mean(results), mse, len(results) =  0.4936031589338598 tf.Tensor(1811.4822, shape=(), dtype=float32) 1013
slice = test, score = 0.4936031589338598

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5276 tf.Tensor(2014.2012, shape=(), dtype=float32) 100
slice = key, score = 0.5276
np.mean(results), mse, len(results) =  0.5248495575221239 tf.Tensor(1985.0392, shape=(), dtype=float32) 2260
slice = train, score = 0.5248495575221239
np.mean(results), mse, len(results) =  0.5035143139190522 tf.Tensor(1954.3757, shape=(), dtype=float32) 1013
slice = test, score = 0.5035143139190522

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.5241 tf.Tensor(2578.0596, shape=(), dtype=float32) 100
slice = key, score = 0.5241
np.mean(results), mse, len(results) =  0.5271814159292034 tf.Tensor(2559.9773, shape=(), dtype=float

np.mean(results), mse, len(results) =  0.5481681415929203 tf.Tensor(4227.831, shape=(), dtype=float32) 2260
slice = train, score = 0.5481681415929203
np.mean(results), mse, len(results) =  0.525538005923001 tf.Tensor(4158.361, shape=(), dtype=float32) 1013
slice = test, score = 0.525538005923001

=== Iteration 36000 ===
np.mean(results), mse, len(results) =  0.5264 tf.Tensor(3529.206, shape=(), dtype=float32) 100
slice = key, score = 0.5264
np.mean(results), mse, len(results) =  0.5365840707964601 tf.Tensor(3463.5522, shape=(), dtype=float32) 2260
slice = train, score = 0.5365840707964601
np.mean(results), mse, len(results) =  0.5126850937808491 tf.Tensor(3400.2397, shape=(), dtype=float32) 1013
slice = test, score = 0.5126850937808491

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.5376 tf.Tensor(4367.043, shape=(), dtype=float32) 100
slice = key, score = 0.5376
np.mean(results), mse, len(results) =  0.5395884955752211 tf.Tensor(4278.0454, shape=(), dtype=float32) 22

np.mean(results), mse, len(results) =  0.5371964461994078 tf.Tensor(4129.001, shape=(), dtype=float32) 1013
slice = test, score = 0.5371964461994078

=== Iteration 54000 ===
np.mean(results), mse, len(results) =  0.5606 tf.Tensor(5001.843, shape=(), dtype=float32) 100
slice = key, score = 0.5606
np.mean(results), mse, len(results) =  0.5621548672566372 tf.Tensor(4940.3657, shape=(), dtype=float32) 2260
slice = train, score = 0.5621548672566372
np.mean(results), mse, len(results) =  0.5325863770977295 tf.Tensor(4859.2764, shape=(), dtype=float32) 1013
slice = test, score = 0.5325863770977295

=== Iteration 55000 ===
np.mean(results), mse, len(results) =  0.5688 tf.Tensor(5824.9697, shape=(), dtype=float32) 100
slice = key, score = 0.5688
np.mean(results), mse, len(results) =  0.5701017699115044 tf.Tensor(5738.0576, shape=(), dtype=float32) 2260
slice = train, score = 0.5701017699115044
np.mean(results), mse, len(results) =  0.5384995064165844 tf.Tensor(5656.897, shape=(), dtype=float32)

np.mean(results), mse, len(results) =  0.5453800592300099 tf.Tensor(4843.359, shape=(), dtype=float32) 1013
slice = test, score = 0.5453800592300099

=== Iteration 72000 ===
np.mean(results), mse, len(results) =  0.5635000000000001 tf.Tensor(5547.3486, shape=(), dtype=float32) 100
slice = key, score = 0.5635000000000001
np.mean(results), mse, len(results) =  0.5729469026548673 tf.Tensor(5534.023, shape=(), dtype=float32) 2260
slice = train, score = 0.5729469026548673
np.mean(results), mse, len(results) =  0.5427640671273445 tf.Tensor(5457.947, shape=(), dtype=float32) 1013
slice = test, score = 0.5427640671273445

=== Iteration 73000 ===
np.mean(results), mse, len(results) =  0.5680000000000001 tf.Tensor(6083.367, shape=(), dtype=float32) 100
slice = key, score = 0.5680000000000001
np.mean(results), mse, len(results) =  0.5693451327433627 tf.Tensor(6114.0293, shape=(), dtype=float32) 2260
slice = train, score = 0.5693451327433627
np.mean(results), mse, len(results) =  0.537769002961500

np.mean(results), mse, len(results) =  0.5450246791707799 tf.Tensor(4891.4053, shape=(), dtype=float32) 1013
slice = test, score = 0.5450246791707799

=== Iteration 90000 ===
np.mean(results), mse, len(results) =  0.5720000000000001 tf.Tensor(4270.148, shape=(), dtype=float32) 100
slice = key, score = 0.5720000000000001
np.mean(results), mse, len(results) =  0.5819690265486726 tf.Tensor(4388.9644, shape=(), dtype=float32) 2260
slice = train, score = 0.5819690265486726
np.mean(results), mse, len(results) =  0.5495459032576506 tf.Tensor(4310.7593, shape=(), dtype=float32) 1013
slice = test, score = 0.5495459032576506

=== Iteration 91000 ===
np.mean(results), mse, len(results) =  0.5796 tf.Tensor(4157.5586, shape=(), dtype=float32) 100
slice = key, score = 0.5796
np.mean(results), mse, len(results) =  0.5802035398230089 tf.Tensor(4201.5977, shape=(), dtype=float32) 2260
slice = train, score = 0.5802035398230089
np.mean(results), mse, len(results) =  0.548568608094768 tf.Tensor(4155.5654,

In [32]:
%%time

ma = AnnCUR(ctx)
ma.fit()
ma.get_score("test")

N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0001,
    "normalize_embs": False,
    "Winit": "eye"
})

m.fit()

np.mean(results), mse, len(results) =  0.5617472852912143 0.146082 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.08289999999999999 tf.Tensor(76.93085083218959, shape=(), dtype=float64) 100
slice = key, score = 0.08289999999999999
np.mean(results), mse, len(results) =  0.08615044247787611 tf.Tensor(77.36240166068123, shape=(), dtype=float64) 2260
slice = train, score = 0.08615044247787611
np.mean(results), mse, len(results) =  0.08596248766041462 tf.Tensor(77.3271918493456, shape=(), dtype=float64) 1013
slice = test, score = 0.08596248766041462

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.42430000000000007 tf.Tensor(98.51324745621396, shape=(), dtype=float64) 100
slice = key, score = 0.424300000

np.mean(results), mse, len(results) =  0.5435752212389381 tf.Tensor(394.9720812939174, shape=(), dtype=float64) 2260
slice = train, score = 0.5435752212389381
np.mean(results), mse, len(results) =  0.5226258637709774 tf.Tensor(403.1895783365347, shape=(), dtype=float64) 1013
slice = test, score = 0.5226258637709774

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.5471000000000001 tf.Tensor(471.1907490475447, shape=(), dtype=float64) 100
slice = key, score = 0.5471000000000001
np.mean(results), mse, len(results) =  0.5480530973451327 tf.Tensor(447.2155605124073, shape=(), dtype=float64) 2260
slice = train, score = 0.5480530973451327
np.mean(results), mse, len(results) =  0.5263376110562685 tf.Tensor(456.6817172870518, shape=(), dtype=float64) 1013
slice = test, score = 0.5263376110562685

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5467 tf.Tensor(521.7868597344749, shape=(), dtype=float64) 100
slice = key, score = 0.5467
np.mean(results), mse, len(r

np.mean(results), mse, len(results) =  0.5415301085883515 tf.Tensor(834.0509340256679, shape=(), dtype=float64) 1013
slice = test, score = 0.5415301085883515

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.5628 tf.Tensor(888.6904995553243, shape=(), dtype=float64) 100
slice = key, score = 0.5628
np.mean(results), mse, len(results) =  0.5670973451327433 tf.Tensor(827.577543875396, shape=(), dtype=float64) 2260
slice = train, score = 0.5670973451327433
np.mean(results), mse, len(results) =  0.5407502467917078 tf.Tensor(850.9599144202609, shape=(), dtype=float64) 1013
slice = test, score = 0.5407502467917078

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.5674 tf.Tensor(882.4600364985786, shape=(), dtype=float64) 100
slice = key, score = 0.5674
np.mean(results), mse, len(results) =  0.5695398230088496 tf.Tensor(819.3585254789641, shape=(), dtype=float64) 2260
slice = train, score = 0.5695398230088496
np.mean(results), mse, len(results) =  0.543326752221

np.mean(results), mse, len(results) =  0.5482625863770978 tf.Tensor(897.8804026516848, shape=(), dtype=float64) 1013
slice = test, score = 0.5482625863770978

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.5695 tf.Tensor(967.6200958314613, shape=(), dtype=float64) 100
slice = key, score = 0.5695
np.mean(results), mse, len(results) =  0.5719911504424778 tf.Tensor(885.415895491437, shape=(), dtype=float64) 2260
slice = train, score = 0.5719911504424778
np.mean(results), mse, len(results) =  0.5441855873642646 tf.Tensor(916.4692487904805, shape=(), dtype=float64) 1013
slice = test, score = 0.5441855873642646

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.5707999999999999 tf.Tensor(939.2070498084033, shape=(), dtype=float64) 100
slice = key, score = 0.5707999999999999
np.mean(results), mse, len(results) =  0.5787300884955752 tf.Tensor(856.9443512199185, shape=(), dtype=float64) 2260
slice = train, score = 0.5787300884955752
np.mean(results), mse, len(re

np.mean(results), mse, len(results) =  0.5536031589338598 tf.Tensor(835.6516101900903, shape=(), dtype=float64) 1013
slice = test, score = 0.5536031589338598

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.5827 tf.Tensor(971.1431061836385, shape=(), dtype=float64) 100
slice = key, score = 0.5827
np.mean(results), mse, len(results) =  0.5865884955752213 tf.Tensor(874.7619709139647, shape=(), dtype=float64) 2260
slice = train, score = 0.5865884955752213
np.mean(results), mse, len(results) =  0.5559526159921027 tf.Tensor(911.5206936161804, shape=(), dtype=float64) 1013
slice = test, score = 0.5559526159921027

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.5779000000000001 tf.Tensor(950.3676125206725, shape=(), dtype=float64) 100
slice = key, score = 0.5779000000000001
np.mean(results), mse, len(results) =  0.5809778761061947 tf.Tensor(852.7325550505237, shape=(), dtype=float64) 2260
slice = train, score = 0.5809778761061947
np.mean(results), mse, len(r

np.mean(results), mse, len(results) =  0.5549950641658441 tf.Tensor(969.8444622877116, shape=(), dtype=float64) 1013
slice = test, score = 0.5549950641658441

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.5847000000000001 tf.Tensor(1087.3604902712593, shape=(), dtype=float64) 100
slice = key, score = 0.5847000000000001
np.mean(results), mse, len(results) =  0.5898362831858407 tf.Tensor(976.5040297807723, shape=(), dtype=float64) 2260
slice = train, score = 0.5898362831858407
np.mean(results), mse, len(results) =  0.5577393879565646 tf.Tensor(1018.7325938788745, shape=(), dtype=float64) 1013
slice = test, score = 0.5577393879565646

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.5816 tf.Tensor(1052.5583142222902, shape=(), dtype=float64) 100
slice = key, score = 0.5816
np.mean(results), mse, len(results) =  0.5881283185840708 tf.Tensor(941.068872401968, shape=(), dtype=float64) 2260
slice = train, score = 0.5881283185840708
np.mean(results), mse, len

In [142]:
%%time
N = 1000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 100,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()

self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.028099999999999997 tf.Tensor(39.920662, shape=(), dtype=float32) 100
slice = key, score = 0.028099999999999997
np.mean(results), mse, len(results) =  0.029216814159292035 tf.Tensor(38.315723, shape=(), dtype=float32) 2260
slice = train, score = 0.029216814159292035
np.mean(results), mse, len(results) =  0.02831194471865745 tf.Tensor(38.145824, shape=(), dtype=float32) 1013
slice = test, score = 0.02831194471865745

=== Iteration 100 ===
np.mean(results), mse, len(results) =  0.35679999999999995 tf.Tensor(64.72958, shape=(), dtype=float32) 100
slice = key, score = 0.35679999999999995
np.mean(results), mse, len(results) =  0.3628274336283186 tf.Tensor(57.514206, shape=(), dty

In [143]:
2*60+33,2*60+54, 3*60+18

(153, 174, 198)

In [144]:
155/198.

0.7828282828282829

In [72]:
isinstance(m.recommend("test"), tf.Tensor)

True

In [73]:
m.recommend("test").argsort

AttributeError: 'tensorflow.python.framework.ops.EagerTensor' object has no attribute 'argsort'

In [77]:
tf.argsort(m.recommend("test"), axis=-1)[:, :100].numpy()

array([[ 9564,  5066,  6927, ...,  6496,  3665,  4425],
       [ 3983,  4579,  7592, ...,  8019,  3948,   784],
       [ 2480,  2987,  4685, ...,  6547,  7153,  5153],
       ...,
       [ 7564,  1609,  9842, ...,  2599,  5220,  3554],
       [ 5223,   475,  7524, ...,  1093,  9880,  1893],
       [ 5223,   358,   475, ...,  5440, 10012,  8682]], dtype=int32)

In [145]:
m = Popular(ctx)

In [146]:
m.fit()

In [147]:
m.get_score("test")

np.mean(results), mse, len(results) =  0.09171767028627838 0.6754948 1013


0.09171767028627838

In [164]:
s = """

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0029000000000000002 tf.Tensor(38.31489, shape=(), dtype=float32) 100
slice = key, score = 0.0029000000000000002
np.mean(results), mse, len(results) =  0.005142571285642822 tf.Tensor(37.841095, shape=(), dtype=float32) 1999
slice = train, score = 0.005142571285642822
np.mean(results), mse, len(results) =  0.0037333333333333337 tf.Tensor(37.865456, shape=(), dtype=float32) 900
slice = test, score = 0.0037333333333333337

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5466000000000001 tf.Tensor(51.295834, shape=(), dtype=float32) 100
slice = key, score = 0.5466000000000001
np.mean(results), mse, len(results) =  0.6161480740370185 tf.Tensor(53.055893, shape=(), dtype=float32) 1999
slice = train, score = 0.6161480740370185
np.mean(results), mse, len(results) =  0.5730555555555555 tf.Tensor(53.265755, shape=(), dtype=float32) 900
slice = test, score = 0.5730555555555555

=== Iteration 2000 ===
np.mean(results), mse, len(results) =  0.5746 tf.Tensor(72.67692, shape=(), dtype=float32) 100
slice = key, score = 0.5746
np.mean(results), mse, len(results) =  0.6468784392196097 tf.Tensor(74.443, shape=(), dtype=float32) 1999
slice = train, score = 0.6468784392196097
np.mean(results), mse, len(results) =  0.6011555555555556 tf.Tensor(74.68854, shape=(), dtype=float32) 900
slice = test, score = 0.6011555555555556

=== Iteration 3000 ===
np.mean(results), mse, len(results) =  0.5852 tf.Tensor(94.415436, shape=(), dtype=float32) 100
slice = key, score = 0.5852
np.mean(results), mse, len(results) =  0.6573536768384192 tf.Tensor(96.217834, shape=(), dtype=float32) 1999
slice = train, score = 0.6573536768384192
np.mean(results), mse, len(results) =  0.6107222222222222 tf.Tensor(96.82078, shape=(), dtype=float32) 900
slice = test, score = 0.6107222222222222

=== Iteration 4000 ===
np.mean(results), mse, len(results) =  0.5871 tf.Tensor(115.53131, shape=(), dtype=float32) 100
slice = key, score = 0.5871
np.mean(results), mse, len(results) =  0.6637718859429715 tf.Tensor(117.33133, shape=(), dtype=float32) 1999
slice = train, score = 0.6637718859429715
np.mean(results), mse, len(results) =  0.6128 tf.Tensor(118.32285, shape=(), dtype=float32) 900
slice = test, score = 0.6128

=== Iteration 5000 ===
np.mean(results), mse, len(results) =  0.5897000000000001 tf.Tensor(135.51602, shape=(), dtype=float32) 100
slice = key, score = 0.5897000000000001
np.mean(results), mse, len(results) =  0.6677538769384693 tf.Tensor(137.80699, shape=(), dtype=float32) 1999
slice = train, score = 0.6677538769384693
np.mean(results), mse, len(results) =  0.6144555555555555 tf.Tensor(138.86089, shape=(), dtype=float32) 900
slice = test, score = 0.6144555555555555

=== Iteration 6000 ===
np.mean(results), mse, len(results) =  0.5989 tf.Tensor(153.00684, shape=(), dtype=float32) 100
slice = key, score = 0.5989
np.mean(results), mse, len(results) =  0.6731815907953977 tf.Tensor(155.92157, shape=(), dtype=float32) 1999
slice = train, score = 0.6731815907953977
np.mean(results), mse, len(results) =  0.6178333333333332 tf.Tensor(157.23227, shape=(), dtype=float32) 900
slice = test, score = 0.6178333333333332

=== Iteration 7000 ===
np.mean(results), mse, len(results) =  0.5921 tf.Tensor(169.03706, shape=(), dtype=float32) 100
slice = key, score = 0.5921
np.mean(results), mse, len(results) =  0.6749574787393697 tf.Tensor(172.92326, shape=(), dtype=float32) 1999
slice = train, score = 0.6749574787393697
np.mean(results), mse, len(results) =  0.6168111111111112 tf.Tensor(174.61946, shape=(), dtype=float32) 900
slice = test, score = 0.6168111111111112

=== Iteration 8000 ===
np.mean(results), mse, len(results) =  0.5934000000000001 tf.Tensor(188.65756, shape=(), dtype=float32) 100
slice = key, score = 0.5934000000000001
np.mean(results), mse, len(results) =  0.6779489744872436 tf.Tensor(190.43488, shape=(), dtype=float32) 1999
slice = train, score = 0.6779489744872436
np.mean(results), mse, len(results) =  0.6186555555555555 tf.Tensor(191.5736, shape=(), dtype=float32) 900
slice = test, score = 0.6186555555555555

=== Iteration 9000 ===
np.mean(results), mse, len(results) =  0.5926000000000001 tf.Tensor(203.36227, shape=(), dtype=float32) 100
slice = key, score = 0.5926000000000001
np.mean(results), mse, len(results) =  0.678304152076038 tf.Tensor(204.18268, shape=(), dtype=float32) 1999
slice = train, score = 0.678304152076038
np.mean(results), mse, len(results) =  0.6191555555555556 tf.Tensor(206.51407, shape=(), dtype=float32) 900
slice = test, score = 0.6191555555555556

=== Iteration 10000 ===
np.mean(results), mse, len(results) =  0.5981000000000001 tf.Tensor(216.21275, shape=(), dtype=float32) 100
slice = key, score = 0.5981000000000001
np.mean(results), mse, len(results) =  0.6803401700850426 tf.Tensor(219.6277, shape=(), dtype=float32) 1999
slice = train, score = 0.6803401700850426
np.mean(results), mse, len(results) =  0.6189888888888889 tf.Tensor(222.18898, shape=(), dtype=float32) 900
slice = test, score = 0.6189888888888889

=== Iteration 11000 ===
np.mean(results), mse, len(results) =  0.5946000000000001 tf.Tensor(231.79863, shape=(), dtype=float32) 100
slice = key, score = 0.5946000000000001
np.mean(results), mse, len(results) =  0.6821160580290145 tf.Tensor(235.13744, shape=(), dtype=float32) 1999
slice = train, score = 0.6821160580290145
np.mean(results), mse, len(results) =  0.6217666666666666 tf.Tensor(237.82777, shape=(), dtype=float32) 900
slice = test, score = 0.6217666666666666

=== Iteration 12000 ===
np.mean(results), mse, len(results) =  0.5937 tf.Tensor(243.55502, shape=(), dtype=float32) 100
slice = key, score = 0.5937
np.mean(results), mse, len(results) =  0.6844272136068035 tf.Tensor(247.14418, shape=(), dtype=float32) 1999
slice = train, score = 0.6844272136068035
np.mean(results), mse, len(results) =  0.6211888888888888 tf.Tensor(249.12077, shape=(), dtype=float32) 900
slice = test, score = 0.6211888888888888

=== Iteration 13000 ===
np.mean(results), mse, len(results) =  0.5957000000000001 tf.Tensor(257.5348, shape=(), dtype=float32) 100
slice = key, score = 0.5957000000000001
np.mean(results), mse, len(results) =  0.6850225112556277 tf.Tensor(260.61337, shape=(), dtype=float32) 1999
slice = train, score = 0.6850225112556277
np.mean(results), mse, len(results) =  0.6214 tf.Tensor(262.32715, shape=(), dtype=float32) 900
slice = test, score = 0.6214

=== Iteration 14000 ===
np.mean(results), mse, len(results) =  0.5963 tf.Tensor(268.1763, shape=(), dtype=float32) 100
slice = key, score = 0.5963
np.mean(results), mse, len(results) =  0.6869334667333666 tf.Tensor(271.45035, shape=(), dtype=float32) 1999
slice = train, score = 0.6869334667333666
np.mean(results), mse, len(results) =  0.6220666666666665 tf.Tensor(274.22897, shape=(), dtype=float32) 900
slice = test, score = 0.6220666666666665

=== Iteration 15000 ===
np.mean(results), mse, len(results) =  0.5915 tf.Tensor(278.70688, shape=(), dtype=float32) 100
slice = key, score = 0.5915
np.mean(results), mse, len(results) =  0.6882141070535268 tf.Tensor(283.78772, shape=(), dtype=float32) 1999
slice = train, score = 0.6882141070535268
np.mean(results), mse, len(results) =  0.6238222222222223 tf.Tensor(286.7364, shape=(), dtype=float32) 900
slice = test, score = 0.6238222222222223

=== Iteration 16000 ===
np.mean(results), mse, len(results) =  0.6008000000000001 tf.Tensor(291.46405, shape=(), dtype=float32) 100
slice = key, score = 0.6008000000000001
np.mean(results), mse, len(results) =  0.6891845922961481 tf.Tensor(294.50818, shape=(), dtype=float32) 1999
slice = train, score = 0.6891845922961481
np.mean(results), mse, len(results) =  0.6228222222222223 tf.Tensor(298.33755, shape=(), dtype=float32) 900
slice = test, score = 0.6228222222222223

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.5986 tf.Tensor(297.10608, shape=(), dtype=float32) 100
slice = key, score = 0.5986

np.mean(results), mse, len(results) =  0.6902851425712856 tf.Tensor(303.21515, shape=(), dtype=float32) 1999
slice = train, score = 0.6902851425712856
np.mean(results), mse, len(results) =  0.6231333333333333 tf.Tensor(307.59198, shape=(), dtype=float32) 900
slice = test, score = 0.6231333333333333

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6007000000000001 tf.Tensor(310.74814, shape=(), dtype=float32) 100
slice = key, score = 0.6007000000000001
np.mean(results), mse, len(results) =  0.6909854927463732 tf.Tensor(315.59665, shape=(), dtype=float32) 1999
slice = train, score = 0.6909854927463732
np.mean(results), mse, len(results) =  0.6236555555555555 tf.Tensor(320.10052, shape=(), dtype=float32) 900
slice = test, score = 0.6236555555555555

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.598 tf.Tensor(319.99103, shape=(), dtype=float32) 100
slice = key, score = 0.598
np.mean(results), mse, len(results) =  0.6919959979989995 tf.Tensor(325.6205, shape=(), dtype=float32) 1999
slice = train, score = 0.6919959979989995
np.mean(results), mse, len(results) =  0.6222111111111112 tf.Tensor(330.26843, shape=(), dtype=float32) 900
slice = test, score = 0.6222111111111112

=== Iteration 20000 ===
np.mean(results), mse, len(results) =  0.5932 tf.Tensor(330.1853, shape=(), dtype=float32) 100
slice = key, score = 0.5932
np.mean(results), mse, len(results) =  0.6924012006003002 tf.Tensor(334.11005, shape=(), dtype=float32) 1999
slice = train, score = 0.6924012006003002
np.mean(results), mse, len(results) =  0.6217555555555555 tf.Tensor(339.55258, shape=(), dtype=float32) 900
slice = test, score = 0.6217555555555555

=== Iteration 21000 ===
np.mean(results), mse, len(results) =  0.5974 tf.Tensor(329.55414, shape=(), dtype=float32) 100
slice = key, score = 0.5974
np.mean(results), mse, len(results) =  0.6930115057528764 tf.Tensor(337.73785, shape=(), dtype=float32) 1999
slice = train, score = 0.6930115057528764
np.mean(results), mse, len(results) =  0.6236777777777777 tf.Tensor(342.25284, shape=(), dtype=float32) 900
slice = test, score = 0.6236777777777777

=== Iteration 22000 ===
np.mean(results), mse, len(results) =  0.5992000000000001 tf.Tensor(339.55212, shape=(), dtype=float32) 100
slice = key, score = 0.5992000000000001
np.mean(results), mse, len(results) =  0.694392196098049 tf.Tensor(347.96072, shape=(), dtype=float32) 1999
slice = train, score = 0.694392196098049
np.mean(results), mse, len(results) =  0.6224 tf.Tensor(353.4459, shape=(), dtype=float32) 900
slice = test, score = 0.6224

=== Iteration 23000 ===
np.mean(results), mse, len(results) =  0.5969 tf.Tensor(351.84885, shape=(), dtype=float32) 100
slice = key, score = 0.5969
np.mean(results), mse, len(results) =  0.6945522761380689 tf.Tensor(356.78296, shape=(), dtype=float32) 1999
slice = train, score = 0.6945522761380689
np.mean(results), mse, len(results) =  0.6227888888888888 tf.Tensor(362.68744, shape=(), dtype=float32) 900
slice = test, score = 0.6227888888888888

=== Iteration 24000 ===
np.mean(results), mse, len(results) =  0.5985 tf.Tensor(355.6144, shape=(), dtype=float32) 100
slice = key, score = 0.5985
np.mean(results), mse, len(results) =  0.6961630815407703 tf.Tensor(364.33978, shape=(), dtype=float32) 1999
slice = train, score = 0.6961630815407703
np.mean(results), mse, len(results) =  0.6237444444444444 tf.Tensor(368.58984, shape=(), dtype=float32) 900
slice = test, score = 0.6237444444444444

=== Iteration 25000 ===
np.mean(results), mse, len(results) =  0.5973000000000002 tf.Tensor(363.7288, shape=(), dtype=float32) 100
slice = key, score = 0.5973000000000002
np.mean(results), mse, len(results) =  0.6968584292146073 tf.Tensor(372.26443, shape=(), dtype=float32) 1999
slice = train, score = 0.6968584292146073
np.mean(results), mse, len(results) =  0.6227333333333332 tf.Tensor(377.97552, shape=(), dtype=float32) 900
slice = test, score = 0.6227333333333332

=== Iteration 26000 ===
np.mean(results), mse, len(results) =  0.5955999999999999 tf.Tensor(365.85416, shape=(), dtype=float32) 100
slice = key, score = 0.5955999999999999
np.mean(results), mse, len(results) =  0.6971985992996499 tf.Tensor(373.0473, shape=(), dtype=float32) 1999
slice = train, score = 0.6971985992996499
np.mean(results), mse, len(results) =  0.6224888888888889 tf.Tensor(380.26498, shape=(), dtype=float32) 900
slice = test, score = 0.6224888888888889

=== Iteration 27000 ===
np.mean(results), mse, len(results) =  0.6008 tf.Tensor(372.46814, shape=(), dtype=float32) 100
slice = key, score = 0.6008
np.mean(results), mse, len(results) =  0.6980990495247624 tf.Tensor(378.57083, shape=(), dtype=float32) 1999
slice = train, score = 0.6980990495247624
np.mean(results), mse, len(results) =  0.6238333333333334 tf.Tensor(383.79016, shape=(), dtype=float32) 900
slice = test, score = 0.6238333333333334

=== Iteration 28000 ===
np.mean(results), mse, len(results) =  0.598 tf.Tensor(389.34348, shape=(), dtype=float32) 100
slice = key, score = 0.598
np.mean(results), mse, len(results) =  0.6984592296148076 tf.Tensor(393.3084, shape=(), dtype=float32) 1999
slice = train, score = 0.6984592296148076
np.mean(results), mse, len(results) =  0.6238777777777778 tf.Tensor(398.197, shape=(), dtype=float32) 900
slice = test, score = 0.6238777777777778

=== Iteration 29000 ===
np.mean(results), mse, len(results) =  0.6003 tf.Tensor(388.90027, shape=(), dtype=float32) 100
slice = key, score = 0.6003
np.mean(results), mse, len(results) =  0.6994647323661831 tf.Tensor(394.62967, shape=(), dtype=float32) 1999
slice = train, score = 0.6994647323661831
np.mean(results), mse, len(results) =  0.6231222222222221 tf.Tensor(400.3588, shape=(), dtype=float32) 900
slice = test, score = 0.6231222222222221

=== Iteration 30000 ===
np.mean(results), mse, len(results) =  0.5956000000000001 tf.Tensor(402.19598, shape=(), dtype=float32) 100
slice = key, score = 0.5956000000000001
np.mean(results), mse, len(results) =  0.7004652326163081 tf.Tensor(406.87015, shape=(), dtype=float32) 1999
slice = train, score = 0.7004652326163081
np.mean(results), mse, len(results) =  0.6228555555555556 tf.Tensor(413.48285, shape=(), dtype=float32) 900
slice = test, score = 0.6228555555555556

=== Iteration 31000 ===
np.mean(results), mse, len(results) =  0.5982999999999999 tf.Tensor(404.54608, shape=(), dtype=float32) 100
slice = key, score = 0.5982999999999999
np.mean(results), mse, len(results) =  0.6999849924962481 tf.Tensor(409.2368, shape=(), dtype=float32) 1999
slice = train, score = 0.6999849924962481
np.mean(results), mse, len(results) =  0.6243666666666667 tf.Tensor(415.25107, shape=(), dtype=float32) 900
slice = test, score = 0.6243666666666667

=== Iteration 32000 ===
np.mean(results), mse, len(results) =  0.5988 tf.Tensor(404.69724, shape=(), dtype=float32) 100
slice = key, score = 0.5988
np.mean(results), mse, len(results) =  0.7002851425712856 tf.Tensor(412.9996, shape=(), dtype=float32) 1999
slice = train, score = 0.7002851425712856
np.mean(results), mse, len(results) =  0.6237222222222223 tf.Tensor(419.58795, shape=(), dtype=float32) 900
slice = test, score = 0.6237222222222223

=== Iteration 33000 ===
np.mean(results), mse, len(results) =  0.5892000000000001 tf.Tensor(412.2223, shape=(), dtype=float32) 100
slice = key, score = 0.5892000000000001
np.mean(results), mse, len(results) =  0.7018159079539769 tf.Tensor(418.07166, shape=(), dtype=float32) 1999
slice = train, score = 0.7018159079539769
np.mean(results), mse, len(results) =  0.6230777777777777 tf.Tensor(424.47766, shape=(), dtype=float32) 900
slice = test, score = 0.6230777777777777

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.5974 tf.Tensor(421.2193, shape=(), dtype=float32) 100
slice = key, score = 0.5974
np.mean(results), mse, len(results) =  0.7015957978989494 tf.Tensor(427.40485, shape=(), dtype=float32) 1999
slice = train, score = 0.7015957978989494
np.mean(results), mse, len(results) =  0.6239 tf.Tensor(433.34784, shape=(), dtype=float32) 900
slice = test, score = 0.6239

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.5969 tf.Tensor(425.78943, shape=(), dtype=float32) 100
slice = key, score = 0.5969
np.mean(results), mse, len(results) =  0.7021210605302651 tf.Tensor(431.59137, shape=(), dtype=float32) 1999
slice = train, score = 0.7021210605302651

np.mean(results), mse, len(results) =  0.6236666666666666 tf.Tensor(438.43817, shape=(), dtype=float32) 900
slice = test, score = 0.6236666666666666

=== Iteration 36000 ===
np.mean(results), mse, len(results) =  0.5973 tf.Tensor(425.03436, shape=(), dtype=float32) 100
slice = key, score = 0.5973
np.mean(results), mse, len(results) =  0.704112056028014 tf.Tensor(432.186, shape=(), dtype=float32) 1999
slice = train, score = 0.704112056028014
np.mean(results), mse, len(results) =  0.6254111111111111 tf.Tensor(438.95386, shape=(), dtype=float32) 900
slice = test, score = 0.6254111111111111

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.5961 tf.Tensor(437.49792, shape=(), dtype=float32) 100
slice = key, score = 0.5961
np.mean(results), mse, len(results) =  0.7033016508254127 tf.Tensor(441.19547, shape=(), dtype=float32) 1999
slice = train, score = 0.7033016508254127
np.mean(results), mse, len(results) =  0.6231333333333334 tf.Tensor(448.3536, shape=(), dtype=float32) 900
slice = test, score = 0.6231333333333334

=== Iteration 38000 ===
np.mean(results), mse, len(results) =  0.5943 tf.Tensor(443.49838, shape=(), dtype=float32) 100
slice = key, score = 0.5943
np.mean(results), mse, len(results) =  0.7041420710355177 tf.Tensor(448.03815, shape=(), dtype=float32) 1999
slice = train, score = 0.7041420710355177
np.mean(results), mse, len(results) =  0.6240555555555556 tf.Tensor(454.8092, shape=(), dtype=float32) 900
slice = test, score = 0.6240555555555556

=== Iteration 39000 ===
np.mean(results), mse, len(results) =  0.5952000000000001 tf.Tensor(449.2005, shape=(), dtype=float32) 100
slice = key, score = 0.5952000000000001
np.mean(results), mse, len(results) =  0.7036568284142072 tf.Tensor(451.87326, shape=(), dtype=float32) 1999
slice = train, score = 0.7036568284142072
np.mean(results), mse, len(results) =  0.6233000000000001 tf.Tensor(458.52914, shape=(), dtype=float32) 900
slice = test, score = 0.6233000000000001

=== Iteration 40000 ===
np.mean(results), mse, len(results) =  0.5942000000000001 tf.Tensor(452.93558, shape=(), dtype=float32) 100
slice = key, score = 0.5942000000000001
np.mean(results), mse, len(results) =  0.703471735867934 tf.Tensor(456.82068, shape=(), dtype=float32) 1999
slice = train, score = 0.703471735867934
np.mean(results), mse, len(results) =  0.622922222222222 tf.Tensor(464.0835, shape=(), dtype=float32) 900
slice = test, score = 0.622922222222222

=== Iteration 41000 ===
np.mean(results), mse, len(results) =  0.6014999999999999 tf.Tensor(457.9872, shape=(), dtype=float32) 100
slice = key, score = 0.6014999999999999
np.mean(results), mse, len(results) =  0.7063081540770384 tf.Tensor(463.51547, shape=(), dtype=float32) 1999
slice = train, score = 0.7063081540770384
np.mean(results), mse, len(results) =  0.6250222222222223 tf.Tensor(469.912, shape=(), dtype=float32) 900
slice = test, score = 0.6250222222222223

=== Iteration 42000 ===
np.mean(results), mse, len(results) =  0.5956 tf.Tensor(466.80145, shape=(), dtype=float32) 100
slice = key, score = 0.5956
np.mean(results), mse, len(results) =  0.7053226613306653 tf.Tensor(468.68228, shape=(), dtype=float32) 1999
slice = train, score = 0.7053226613306653
np.mean(results), mse, len(results) =  0.6250111111111111 tf.Tensor(476.50204, shape=(), dtype=float32) 900
slice = test, score = 0.6250111111111111

=== Iteration 43000 ===
np.mean(results), mse, len(results) =  0.5999 tf.Tensor(471.70297, shape=(), dtype=float32) 100
slice = key, score = 0.5999
np.mean(results), mse, len(results) =  0.705207603801901 tf.Tensor(477.5338, shape=(), dtype=float32) 1999
slice = train, score = 0.705207603801901
np.mean(results), mse, len(results) =  0.6243777777777778 tf.Tensor(485.27557, shape=(), dtype=float32) 900
slice = test, score = 0.6243777777777778

=== Iteration 44000 ===
np.mean(results), mse, len(results) =  0.596 tf.Tensor(474.33582, shape=(), dtype=float32) 100
slice = key, score = 0.596
np.mean(results), mse, len(results) =  0.7062231115557779 tf.Tensor(477.82468, shape=(), dtype=float32) 1999
slice = train, score = 0.7062231115557779
np.mean(results), mse, len(results) =  0.6246555555555556 tf.Tensor(486.6591, shape=(), dtype=float32) 900
slice = test, score = 0.6246555555555556

=== Iteration 45000 ===
np.mean(results), mse, len(results) =  0.5946 tf.Tensor(474.9135, shape=(), dtype=float32) 100
slice = key, score = 0.5946
np.mean(results), mse, len(results) =  0.7065882941470736 tf.Tensor(478.02023, shape=(), dtype=float32) 1999
slice = train, score = 0.7065882941470736
np.mean(results), mse, len(results) =  0.6232666666666667 tf.Tensor(485.45508, shape=(), dtype=float32) 900
slice = test, score = 0.6232666666666667

=== Iteration 46000 ===
np.mean(results), mse, len(results) =  0.5948 tf.Tensor(480.51935, shape=(), dtype=float32) 100
slice = key, score = 0.5948
np.mean(results), mse, len(results) =  0.7062381190595297 tf.Tensor(486.72763, shape=(), dtype=float32) 1999
slice = train, score = 0.7062381190595297
np.mean(results), mse, len(results) =  0.6217666666666667 tf.Tensor(494.45288, shape=(), dtype=float32) 900
slice = test, score = 0.6217666666666667

=== Iteration 47000 ===
np.mean(results), mse, len(results) =  0.5942000000000001 tf.Tensor(487.95572, shape=(), dtype=float32) 100
slice = key, score = 0.5942000000000001
np.mean(results), mse, len(results) =  0.7076138069034518 tf.Tensor(489.15692, shape=(), dtype=float32) 1999
slice = train, score = 0.7076138069034518
np.mean(results), mse, len(results) =  0.6225 tf.Tensor(497.5497, shape=(), dtype=float32) 900
slice = test, score = 0.6225

=== Iteration 48000 ===
np.mean(results), mse, len(results) =  0.5949 tf.Tensor(491.6003, shape=(), dtype=float32) 100
slice = key, score = 0.5949
np.mean(results), mse, len(results) =  0.7077238619309655 tf.Tensor(494.7427, shape=(), dtype=float32) 1999
slice = train, score = 0.7077238619309655
np.mean(results), mse, len(results) =  0.6235888888888889 tf.Tensor(503.38812, shape=(), dtype=float32) 900
slice = test, score = 0.6235888888888889

=== Iteration 49000 ===
np.mean(results), mse, len(results) =  0.5941999999999998 tf.Tensor(493.59476, shape=(), dtype=float32) 100
slice = key, score = 0.5941999999999998
np.mean(results), mse, len(results) =  0.7081240620310155 tf.Tensor(496.56677, shape=(), dtype=float32) 1999
slice = train, score = 0.7081240620310155
np.mean(results), mse, len(results) =  0.6246000000000002 tf.Tensor(505.2944, shape=(), dtype=float32) 900
slice = test, score = 0.6246000000000002

=== Iteration 50000 ===
np.mean(results), mse, len(results) =  0.5929 tf.Tensor(503.98883, shape=(), dtype=float32) 100
slice = key, score = 0.5929
np.mean(results), mse, len(results) =  0.7097098549274637 tf.Tensor(508.66168, shape=(), dtype=float32) 1999
slice = train, score = 0.7097098549274637
np.mean(results), mse, len(results) =  0.6228666666666666 tf.Tensor(518.20984, shape=(), dtype=float32) 900
slice = test, score = 0.6228666666666666

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.5943 tf.Tensor(507.84497, shape=(), dtype=float32) 100
slice = key, score = 0.5943
np.mean(results), mse, len(results) =  0.7082141070535268 tf.Tensor(509.8853, shape=(), dtype=float32) 1999
slice = train, score = 0.7082141070535268
np.mean(results), mse, len(results) =  0.6215 tf.Tensor(517.4809, shape=(), dtype=float32) 900
slice = test, score = 0.6215

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.5977 tf.Tensor(510.63174, shape=(), dtype=float32) 100
slice = key, score = 0.5977
np.mean(results), mse, len(results) =  0.708784392196098 tf.Tensor(517.5887, shape=(), dtype=float32) 1999
slice = train, score = 0.708784392196098
np.mean(results), mse, len(results) =  0.6235111111111111 tf.Tensor(526.492, shape=(), dtype=float32) 900
slice = test, score = 0.6235111111111111

=== Iteration 53000 ===
np.mean(results), mse, len(results) =  0.592 tf.Tensor(515.6737, shape=(), dtype=float32) 100
slice = key, score = 0.592
np.mean(results), mse, len(results) =  0.7090245122561282 tf.Tensor(522.4199, shape=(), dtype=float32) 1999
slice = train, score = 0.7090245122561282
np.mean(results), mse, len(results) =  0.6226333333333334 tf.Tensor(529.9929, shape=(), dtype=float32) 900
slice = test, score = 0.6226333333333334


=== Iteration 54000 ===
np.mean(results), mse, len(results) =  0.5921 tf.Tensor(520.27045, shape=(), dtype=float32) 100
slice = key, score = 0.5921
np.mean(results), mse, len(results) =  0.7091945972986493 tf.Tensor(524.4802, shape=(), dtype=float32) 1999
slice = train, score = 0.7091945972986493
np.mean(results), mse, len(results) =  0.6224888888888889 tf.Tensor(533.34485, shape=(), dtype=float32) 900
slice = test, score = 0.6224888888888889

=== Iteration 55000 ===
np.mean(results), mse, len(results) =  0.5968 tf.Tensor(527.9664, shape=(), dtype=float32) 100
slice = key, score = 0.5968
np.mean(results), mse, len(results) =  0.7095697848924462 tf.Tensor(531.3875, shape=(), dtype=float32) 1999
slice = train, score = 0.7095697848924462
np.mean(results), mse, len(results) =  0.6234666666666667 tf.Tensor(541.15, shape=(), dtype=float32) 900
slice = test, score = 0.6234666666666667

=== Iteration 56000 ===
np.mean(results), mse, len(results) =  0.5957 tf.Tensor(532.022, shape=(), dtype=float32) 100
slice = key, score = 0.5957
np.mean(results), mse, len(results) =  0.7101950975487743 tf.Tensor(535.8775, shape=(), dtype=float32) 1999
slice = train, score = 0.7101950975487743
np.mean(results), mse, len(results) =  0.6218777777777779 tf.Tensor(545.3727, shape=(), dtype=float32) 900
slice = test, score = 0.6218777777777779

=== Iteration 57000 ===
np.mean(results), mse, len(results) =  0.5926999999999999 tf.Tensor(537.7481, shape=(), dtype=float32) 100
slice = key, score = 0.5926999999999999
np.mean(results), mse, len(results) =  0.709144572286143 tf.Tensor(539.824, shape=(), dtype=float32) 1999
slice = train, score = 0.709144572286143
np.mean(results), mse, len(results) =  0.6219666666666667 tf.Tensor(550.64795, shape=(), dtype=float32) 900
slice = test, score = 0.6219666666666667

=== Iteration 58000 ===
np.mean(results), mse, len(results) =  0.5985 tf.Tensor(541.64813, shape=(), dtype=float32) 100
slice = key, score = 0.5985
np.mean(results), mse, len(results) =  0.7100650325162582 tf.Tensor(543.7328, shape=(), dtype=float32) 1999
slice = train, score = 0.7100650325162582
np.mean(results), mse, len(results) =  0.6214888888888888 tf.Tensor(554.3544, shape=(), dtype=float32) 900
slice = test, score = 0.6214888888888888


"""

In [165]:
def get_train_test(s):
    v = s.strip().split("\n")
    try:
        key = [v_i for v_i in v if v_i.startswith("slice = key")][0]
        train = [v_i for v_i in v if v_i.startswith("slice = train")][0]
        test = [v_i for v_i in v if v_i.startswith("slice = test")][0]
        return float(key.split(" = ")[-1]), float(train.split(" = ")[-1]), float(test.split(" = ")[-1])
    except Exception as e:
        print("wtf")
        return (-1, -1)
    
def get_best(s):
    return sorted([get_train_test(x) for x in s.strip().split("=== Iteration") if x], reverse=True)[:3]

In [166]:
get_best(s)

[(0.6014999999999999, 0.7063081540770384, 0.6250222222222223),
 (0.6008000000000001, 0.6891845922961481, 0.6228222222222223),
 (0.6008, 0.6980990495247624, 0.6238333333333334)]

In [167]:
0.7233923303834808 - get_best(s)[0][-1]

0.09837010816125857